In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Passes de transpilador com IA

Os passes de transpilador com IA são passes que funcionam como substituição direta dos passes "tradicionais" do Qiskit para algumas tarefas de transpilação. Com frequência, eles produzem resultados melhores do que os algoritmos heurísticos existentes (como menor profundidade e contagem de CNOTs), mas também são muito mais rápidos do que algoritmos de otimização como os resolvedores de satisfatibilidade booleana. Os passes de transpilador com IA são executados no seu ambiente local.

> **Note:** Os passes de transpilador com IA estão em status de lançamento beta, sujeitos a alterações.
> Se você tiver comentários ou quiser entrar em contato com a equipe de desenvolvimento, use este [canal do Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Os seguintes passes estão disponíveis atualmente:

**Passes de roteamento**
 - `AIRouting`: Seleção de layout e roteamento de circuito

**Passes de síntese de circuito**
 - `AICliffordSynthesis`: Síntese de circuito Clifford
 - `AILinearFunctionSynthesis`: Síntese de circuito de função linear
 - `AIPermutationSynthesis`: Síntese de circuito de permutação

Para usar os passes de transpilador com IA, primeiro instale o pacote `qiskit-ibm-transpiler`. Visite a [documentação da API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) para obter mais informações sobre as diferentes opções disponíveis.

## Executar os passes de transpilador com IA localmente ou na nuvem
Primeiro instale o `qiskit-ibm-transpiler` com algumas dependências extras da seguinte forma:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Após instalar as dependências extras, o modo padrão para executar os passes de transpilador com IA é usar sua máquina local.

## Passe de roteamento com IA
O passe `AIRouting` atua tanto como estágio de layout quanto como estágio de roteamento. Ele pode ser usado dentro de um `PassManager` da seguinte forma:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Aqui, o `backend` determina para qual mapa de acoplamento rotear, o `optimization_level` (1, 2 ou 3) determina o esforço computacional a ser empregado no processo (valores maiores geralmente produzem melhores resultados, mas levam mais tempo), e o `layout_mode` especifica como tratar a seleção de layout.
O `layout_mode` inclui as seguintes opções:

- `keep`: Respeita o layout definido pelos passes de transpilador anteriores (ou usa o layout trivial se não estiver definido). Tipicamente, é usado apenas quando o circuito deve ser executado em qubits específicos do dispositivo. Com frequência, produz resultados piores porque tem menos espaço para otimização.
- `improve`: Usa o layout definido pelos passes de transpilador anteriores como ponto de partida. É útil quando você tem uma boa estimativa inicial para o layout; por exemplo, para circuitos construídos de forma que sigam aproximadamente o mapa de acoplamento do dispositivo. Também é útil se você quiser experimentar outros passes de layout específicos combinados com o passe `AIRouting`.
- `optimize`: Este é o modo padrão. Funciona melhor para circuitos gerais nos quais você pode não ter boas estimativas de layout. Este modo ignora seleções de layout anteriores.
- `local_mode`: Este sinalizador determina onde o passe `AIRouting` é executado. Se `False`, o `AIRouting` é executado remotamente por meio do Qiskit Transpiler Service. Se `True`, o pacote tenta executar o passe no seu ambiente local, com fallback para o modo em nuvem caso as dependências necessárias não sejam encontradas.

## Passes de síntese de circuito com IA
Os passes de síntese de circuito com IA permitem otimizar partes de diferentes tipos de circuito ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Função Linear](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutação](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Rede de Pauli) por meio de re-síntese. Uma forma típica de usar o passe de síntese é a seguinte: